# Pandas 01 — Loading and first inspection

**What's in here**
- `read_csv` options you actually use (`dtype`, `parse_dates`, `usecols`, `nrows`, `na_values`, `index_col`, separators)
- The first-look checklist: shape, `info()`, `describe()`, missing values, duplicates, cardinality
- Checking that timestamps are really timestamps, and that the sampling interval is what you think
- A reusable `inspect(df)` function to run on any unfamiliar dataset
- Writing data back out

Data: `../data/hourly_power_raw.csv` (messy, as received) and `../data/hourly_power_clean.csv` (tidy).

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 3)

## 1. Reading a CSV — the minimal call

`pd.read_csv(path)` guesses everything. That is fine for a first look, but never trust the guessed dtypes: numbers with a stray string become `object`, and timestamps stay as strings.

In [2]:
raw = pd.read_csv("../data/hourly_power_raw.csv")
raw.head()

,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,region
0,2023-03-30 23:00:00,28555.6,4.97,8.75,0.0,70.16,GB
1,2023-07-16 15:00:00,27602.0,21.96,6.52,523.2,23.19,GB
2,2022-12-25 16:00:00,33773.8,4.15,9.57,18.5,115.79,GB
3,2023-03-10 01:00:00,26151.0,2.06,5.83,0.0,80.50,GB
4,2023-11-09 05:00:00,24584.6,6.58,4.76,0.0,69.49,GB


## 2. `shape`, `dtypes`, `info()`

`info()` gives you dtypes, non-null counts and memory in one shot. Read it column by column: here `time` and `price_eur_mwh` are `object` — both are wrong.

**Interview check:** *"Why is price an object column?"* — because at least one value could not be parsed as a number. Find it before converting.

In [3]:
print(raw.shape)
raw.info()

(17457, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17457 entries, 0 to 17456
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   time             17457 non-null  object 
 1   consumption_mwh  17457 non-null  float64
 2   temp_c           17308 non-null  float64
 3   wind_ms          17457 non-null  float64
 4   solar_wm2        17457 non-null  float64
 5   price_eur_mwh    17457 non-null  object 
 6   region           17457 non-null  object 
dtypes: float64(4), object(3)
memory usage: 954.8+ KB


## 3. `describe()` — numeric and everything

`describe()` defaults to numeric columns. `include="all"` adds counts/unique/top for object columns. Always look at `min` and `max`: sentinels like `-999` show up immediately.

In [4]:
raw.describe().round(2)

,consumption_mwh,temp_c,wind_ms,solar_wm2
count,17457.00,17308.00,17457.00,17457.00
mean,29315.46,6.27,7.30,97.73
std,4208.58,61.13,2.50,155.44
min,18092.90,-999.00,0.00,0.00
25%,26446.50,4.45,5.56,0.00
50%,29671.30,9.92,7.23,0.00
75%,32374.40,15.32,8.96,143.10
max,40824.90,27.74,15.99,794.60


In [5]:
raw.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
time,17457,17442,2022-02-04 07:00:00,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
consumption_mwh,17457.0,NaN,NaN,NaN,29315.46,4208.575,18092.9,26446.5,29671.3,32374.4,40824.9
temp_c,17308.0,NaN,NaN,NaN,6.266,61.128,-999.0,4.45,9.92,15.32,27.74
wind_ms,17457.0,NaN,NaN,NaN,7.299,2.499,0.0,5.56,7.23,8.96,15.99
solar_wm2,17457.0,NaN,NaN,NaN,97.727,155.438,0.0,0.0,0.0,143.1,794.6
price_eur_mwh,17457,9932,missing,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,17457,1,GB,17457,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Pitfall:** `temp_c` has `min = -999`. That is not a temperature, it is a missing-value sentinel that the data provider used. `describe()` is the cheapest way to spot it.

In [6]:
(raw["temp_c"] <= -100).sum(), raw.loc[raw["temp_c"] <= -100, "temp_c"].unique()

(63, array([-999.]))

## 4. Missing values — count and share

`isna().sum()` gives counts; `isna().mean()` gives the fraction, which is what you actually reason about. Note that `"missing"` strings in `price_eur_mwh` are **not** counted as NaN yet.

In [7]:
pd.DataFrame({"n_missing": raw.isna().sum(), "share": raw.isna().mean().round(4)})

,n_missing,share
time,0,0.000
consumption_mwh,0,0.000
temp_c,149,0.009
wind_ms,0,0.000
solar_wm2,0,0.000
price_eur_mwh,0,0.000
region,0,0.000


## 5. Duplicates

`duplicated()` marks the second and later occurrences of identical rows. `duplicated(subset=[key])` is usually what you want: two rows for the same timestamp are duplicates even if a value differs.

In [8]:
print("exact duplicate rows :", raw.duplicated().sum())
print("duplicate timestamps :", raw.duplicated(subset=["time"]).sum())
raw[raw.duplicated(subset=["time"], keep=False)].sort_values("time").head(6)

exact duplicate rows : 15
duplicate timestamps : 15


,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,region
10516,2022-02-04 07:00:00,32724.5,0.53,5.42,41.5,127.95,GB
10089,2022-02-04 07:00:00,32724.5,0.53,5.42,41.5,127.95,GB
1106,2022-03-07 08:00:00,36737.5,-1.14,6.74,157.4,110.76,GB
2865,2022-03-07 08:00:00,36737.5,-1.14,6.74,157.4,110.76,GB
14908,2022-04-28 19:00:00,33698.2,13.29,4.97,0.0,147.07,GB
1974,2022-04-28 19:00:00,33698.2,13.29,4.97,0.0,147.07,GB


## 6. Cardinality — `nunique()` and `value_counts()`

Low cardinality hints at a categorical; `nunique() == 1` means a constant column that carries no information. `value_counts(dropna=False)` shows NaN as its own category — always use `dropna=False` when inspecting.

In [9]:
raw.nunique()

time               17442
consumption_mwh    16497
temp_c              2844
wind_ms             1366
solar_wm2           4062
price_eur_mwh       9932
region                 1
dtype: int64

In [10]:
raw["region"].value_counts(dropna=False)

region
GB    17457
Name: count, dtype: int64

A constant-column detector — worth keeping as a snippet.

In [11]:
constant_cols = [c for c in raw.columns if raw[c].nunique(dropna=False) <= 1]
constant_cols

['region']

## 7. Is the timestamp really a timestamp?

`dtype == object` means strings. Calling `.dt` on strings raises. This is one of the most common silent problems in a notebook handed to you.

In [12]:
print(raw["time"].dtype)
try:
    raw["time"].dt.hour
except AttributeError as e:
    print("AttributeError:", e)

object
AttributeError: Can only use .dt accessor with datetimelike values


Convert with `pd.to_datetime`. Use `utc=True` to get a tz-aware column (avoids DST ambiguity later) and `errors="coerce"` to turn unparseable strings into `NaT` instead of raising — then **count** how many became `NaT`.

In [13]:
t = pd.to_datetime(raw["time"], utc=True, errors="coerce")
print(t.dtype)
print("unparseable:", t.isna().sum())
print(t.min(), "->", t.max())

datetime64[ns, UTC]
unparseable: 0
2022-01-01 00:00:00+00:00 -> 2023-12-31 23:00:00+00:00


## 8. Is the sampling interval constant?

Sort by time, take `diff()`, and look at `value_counts()`. Anything other than a single value means gaps or duplicates.

**Interview check:** *"What is one observation here?"* — one hour of the GB system. *"Are there missing hours?"* — yes: the gaps of 2h+ and the 0h diffs (duplicates) below prove it.

In [14]:
t_sorted = t.sort_values()
t_sorted.diff().value_counts().head(8)

time
0 days 01:00:00    17387
0 days 02:00:00       52
0 days 00:00:00       15
1 days 01:00:00        1
0 days 03:00:00        1
Name: count, dtype: int64

## 9. `read_csv` options worth memorising

| option | use |
|---|---|
| `parse_dates=[...]` | parse columns to datetime while reading |
| `dtype={...}` | force dtypes (e.g. IDs as `str`, not int) |
| `usecols=[...]` | only load the columns you need — big memory saver |
| `nrows=N` | peek at a huge file |
| `na_values=[...]` | extra strings to treat as NaN (`"missing"`, `"-999"`) |
| `index_col=` | set the index while reading |
| `sep=`, `decimal=`, `thousands=` | European CSVs: `sep=";"`, `decimal=","` |
| `chunksize=N` | iterate over a file too big for memory |

In [15]:
df = pd.read_csv(
    "../data/hourly_power_raw.csv",
    usecols=["time", "consumption_mwh", "temp_c", "price_eur_mwh"],
    parse_dates=["time"],
    na_values={"price_eur_mwh": ["missing"], "temp_c": [-999]},
    dtype={"consumption_mwh": "float64"},
)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17457 entries, 0 to 17456
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   time             17457 non-null  datetime64[ns]
 1   consumption_mwh  17457 non-null  float64       
 2   temp_c           17245 non-null  float64       
 3   price_eur_mwh    17357 non-null  float64       
dtypes: datetime64[ns](1), float64(3)
memory usage: 545.7 KB


**Pitfall:** `parse_dates` gives a **tz-naive** datetime here because the raw strings have no offset. Whether the data is in UTC or local time is a question for the data owner, not for pandas. Localise explicitly once you know.

In [16]:
print(df["time"].dtype)
df["time"] = df["time"].dt.tz_localize("UTC")
print(df["time"].dtype)

datetime64[ns]
datetime64[ns, UTC]


`nrows` to peek, `chunksize` to stream. The chunk pattern: aggregate each chunk, then combine.

In [17]:
peek = pd.read_csv("../data/hourly_power_raw.csv", nrows=5)
print(peek.shape)

total_rows = 0
for chunk in pd.read_csv("../data/hourly_power_raw.csv", chunksize=5000):
    total_rows += len(chunk)
total_rows

(5, 7)


17457

## 10. Reading the clean file with timestamps that carry an offset

Strings like `2022-01-01 00:00:00+00:00` parse to `datetime64[ns, UTC]` directly. Confirm the dtype — do not assume.

In [18]:
clean = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
print(clean["time"].dtype)
print(clean.shape)
clean["time"].diff().value_counts()

datetime64[ns, UTC]
(17520, 6)


time
0 days 01:00:00    17519
Name: count, dtype: int64

## 11. Memory

`memory_usage(deep=True)` counts the actual string payload of object columns. Object columns are typically 5–10× more expensive than numeric ones; that is a reason to convert early.

In [19]:
(raw.memory_usage(deep=True) / 1e6).round(2)

Index              0.00
time               1.33
consumption_mwh    0.14
temp_c             0.14
wind_ms            0.14
solar_wm2          0.14
price_eur_mwh      1.09
region             1.03
dtype: float64

## 12. `head` / `tail` / `sample`

`head()` shows you the file order (often meaningless), `tail()` shows how the file ends (truncated? footer rows?), `sample()` shows rows you did not cherry-pick. Use `random_state` so you can reproduce.

In [20]:
raw.sample(5, random_state=0)

,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,region
11821,2022-06-30 13:00:00,29145.0,21.25,6.15,550.9,85.40,GB
5292,2023-07-01 04:00:00,19273.6,15.87,7.59,0.0,23.51,GB
2601,2022-10-23 13:00:00,27032.4,13.94,7.32,185.7,114.39,GB
4388,2022-10-21 12:00:00,30320.7,11.66,11.43,112.3,100.20,GB
10756,2022-06-18 21:00:00,25952.8,16.18,6.55,0.0,111.73,GB


## 13. A reusable `inspect(df)` checklist

Paste this into any new notebook. It answers the first five questions you should ask of unfamiliar data in one call.

In [21]:
def inspect(df, time_col=None, max_unique=10):
    print(f"shape: {df.shape}")
    print(f"memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
    print(f"exact duplicate rows: {df.duplicated().sum()}")
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (100 * df.isna().mean()).round(2),
        "n_unique": df.nunique(dropna=False),
    })
    print(summary.to_string())
    const = summary.index[summary["n_unique"] <= 1].tolist()
    if const:
        print(f"constant columns: {const}")
    obj_cols = df.select_dtypes("object").columns
    for c in obj_cols:
        if df[c].nunique() <= max_unique:
            print(f"\n{c}: {df[c].value_counts(dropna=False).to_dict()}")
    if time_col is not None:
        t = pd.to_datetime(df[time_col], utc=True, errors="coerce")
        print(f"\n{time_col}: {t.min()} -> {t.max()}, unparseable={t.isna().sum()}, "
              f"duplicate stamps={t.duplicated().sum()}")
        print("interval counts:", t.sort_values().diff().value_counts().head(4).to_dict())

inspect(raw, time_col="time")

shape: (17457, 7)
memory: 4.0 MB
exact duplicate rows: 15
                   dtype  n_missing  pct_missing  n_unique
time              object          0         0.00     17442
consumption_mwh  float64          0         0.00     16497
temp_c           float64        149         0.85      2845
wind_ms          float64          0         0.00      1366
solar_wm2        float64          0         0.00      4062
price_eur_mwh     object          0         0.00      9932
region            object          0         0.00         1
constant columns: ['region']



region: {'GB': 17457}

time: 2022-01-01 00:00:00+00:00 -> 2023-12-31 23:00:00+00:00, unparseable=0, duplicate stamps=15
interval counts: {Timedelta('0 days 01:00:00'): 17387, Timedelta('0 days 02:00:00'): 52, Timedelta('0 days 00:00:00'): 15, Timedelta('1 days 01:00:00'): 1}


## 14. Writing data out

`to_csv(index=False)` unless the index is meaningful (a time index usually is — then keep it). Parquet preserves dtypes (including tz-aware timestamps) and is much faster, but needs `pyarrow`.

In [22]:
import importlib.util, os, tempfile

tmp = os.path.join(tempfile.gettempdir(), "clean_copy.csv")
clean.to_csv(tmp, index=False)
back = pd.read_csv(tmp, parse_dates=["time"])
print("round-trip dtype:", back["time"].dtype)
print("pyarrow available:", importlib.util.find_spec("pyarrow") is not None,
      "-> use df.to_parquet(path) / pd.read_parquet(path) when it is")
os.remove(tmp)

round-trip dtype: datetime64[ns, UTC]
pyarrow available: False -> use df.to_parquet(path) / pd.read_parquet(path) when it is


## Summary — questions before any modelling

1. What is one row? (one hour, one meter-day, one trade?)
2. Are the dtypes right — especially timestamps and numeric columns that came in as `object`?
3. How much is missing, per column, and is it random or structured (whole days)?
4. Are there duplicate keys?
5. Is the sampling interval constant? Where are the gaps?
6. Any constant columns, sentinels (`-999`), or impossible values (negative consumption)?